Pytorch 학습 // WINE

Pytorch 모델을 사용하여 wine 데이터를 분류/회기하기
와인 품질 예측하기 -> ['

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
from torchsummary import summary
import numpy as np

In [12]:
# 1. 데이터 로드 (abalone)
path_aba = "/content/drive/MyDrive/가천대학교/2026-1/인공지능개론/Colab Notebooks/2. 데이터/wine.csv"
df = pd.read_csv(path_aba)
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


In [13]:
# 데이터 결측치 확인
df.isnull().sum()

,0
Wine,0
Alcohol,0
Malic.acid,0
Ash,0
Acl,0
Mg,0
Phenols,0
Flavanoids,0
Nonflavanoid.phenols,0
Proanth,0


결측치가 없으므로 라벨 인코딩 진행



In [14]:
# colums 추출?
df.columns # 추출없이 그냥 써도 될듯 함.

# 라벨 인코딩
label_encoders = {}
for column in df.columns:
    label_encoders[column] = LabelEncoder()
    df[column] = label_encoders[column].fit_transform(df[column])
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,0,118,43,44,12,45,73,107,11,87,81,49,120,94
1,0,65,49,18,1,21,68,91,9,31,56,50,102,92
2,0,63,71,63,32,22,73,115,13,96,83,48,86,101
3,0,121,59,49,21,33,95,122,7,84,109,30,104,116
4,0,67,81,75,45,37,73,89,20,66,53,49,73,66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2,92,131,46,42,16,23,12,30,22,108,10,22,67
174,2,76,113,48,51,23,30,19,24,40,104,16,10,68
175,2,68,119,29,40,39,17,17,24,35,125,6,10,74
176,2,64,81,38,40,39,22,16,31,45,120,7,14,75


In [18]:
# 상관관계 분석을 위해 원 핫 인코딩 우선 진행
df_encoded = pd.get_dummies(df, columns=['Alcohol'])
corr_mat=df_encoded.corr()
# 편하게 볼 수 있도록 ascending=False
corr_mat['Alcohol'].sort_values(ascending=False)

KeyError: 'Alcohol'

In [ ]:
# 분석결과를 바탕으로 X(특징) y(정답) 나누기
y_reg = df_encoded['Rings'].values

# Rings 반드시 제외, id는 미리 제거함
X = df_encoded.drop(['Rings'], axis=1)

In [ ]:
# 스케일링 (넘파이로 자동 변환)
scaler_reg = StandardScaler()
X_reg_scaled = scaler_reg.fit_transform(X)

# train / test 분할
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg_scaled, y_reg, test_size=0.2, random_state=0)

In [ ]:
# 딥러닝 회기
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_r.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mse']
)

history = model.fit(
    X_train_r, y_train_r,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 37.5285 - mse: 37.5285 - val_loss: 8.9555 - val_mse: 8.9555
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 6.7906 - mse: 6.7906 - val_loss: 5.5999 - val_mse: 5.5999
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 5.6387 - mse: 5.6387 - val_loss: 5.5814 - val_mse: 5.5814
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 5.2594 - mse: 5.2594 - val_loss: 5.2077 - val_mse: 5.2077
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.9629 - mse: 4.9629 - val_loss: 5.1056 - val_mse: 5.1056
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.8100 - mse: 4.8100 - val_loss: 5.2609 - val_mse: 5.2609
Epoch 7/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.7808 - mse: 4.7808 - val_loss: 5.1608 - val_mse: 5.1608
Epoch 8/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 4.6822 - mse: 4.6822 - val_loss: 4.9899 - val_mse: 4.9899
Epoch 9/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - 

Regression - ['Sex']를 신체지수로 구분하기

In [ ]:
# 분석결과를 바탕으로 X(특징) y(정답) 나누기
y = df['Sex']
# Sex는 반드시 제외, 특징은 치수를 가진 모든 칼럼이 특징이
X = df.drop('Sex', axis=1)

In [ ]:
# y의 순서를 라벨링, 원 핫 인코딩
classes = sorted(y.unique()) # ['F', 'I', 'M'] 랜덤한 순서로 들어왔을 때 착오가 없도록 각자의 위치를 정해둠!
y_encoded = pd.get_dummies(y).reindex(columns=classes, fill_value=0).values

In [ ]:
# 스케일링 (넘파이로 자동 변환)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# train/test 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [ ]:

# 딥러닝 분류
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3), # 과적합 방지
    layers.Dense(32, activation='relu'),
    layers.Dense(y_train.shape[1], activation='softmax') # 3개로 분류 (F, I, M)
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=1
)

Epoch 1/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.5002 - loss: 0.9380 - val_accuracy: 0.5518 - val_loss: 0.8557
Epoch 2/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5389 - loss: 0.8849 - val_accuracy: 0.5470 - val_loss: 0.8447
Epoch 3/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5385 - loss: 0.8800 - val_accuracy: 0.5662 - val_loss: 0.8399
Epoch 4/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5373 - loss: 0.8731 - val_accuracy: 0.5391 - val_loss: 0.8330
Epoch 5/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5549 - loss: 0.8655 - val_accuracy: 0.5742 - val_loss: 0.8362
Epoch 6/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5425 - loss: 0.8669 - val_accuracy: 0.5694 - val_loss: 0.8243
Epoch 7/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5513 - loss: 0.8597 - val_accuracy: 0.5869 - val_loss: 0.8175
Epoch 8/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5585 - loss: 0.8552 - val_accuracy: 0.

In [ ]:
# 성적을 확인하자!
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"최종 성별 판별 정확도: {acc * 100:.2f}%")

최종 성별 판별 정확도: 56.84%


전복의 성별은 외부 형상(길이, 높이, 무게)만으로는 구분하기 어렵다.
50% 구간은 동전의 앞뒷면처럼 찍어서 맞춘 정확도와 다르지 않다고 판단된다..